---
**Regression Analysis in Python**
Data Analysis Course · Week 9
---

This notebook is the Python equivalent of the R Markdown `_08_regression_analysis.Rmd`.
Topics: **simple linear regression**, **multiple regression**, **model evaluation** (RMSE,
train/test split), and using **PCA** to handle correlated predictors.

Work through it cell by cell — run each code cell with **Shift+Enter**.

**Required packages:** `numpy`, `pandas`, `matplotlib`, `seaborn`, `scipy`, `statsmodels`, `scikit-learn`
```
pip install numpy pandas matplotlib seaborn scipy statsmodels scikit-learn
```

We use `statsmodels` for regression here because — like R's `lm()` — it gives a full statistical
summary (coefficients, p-values, R², F-test), not just predictions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.decomposition import PCA

## 1 – Objectives

We build a **regression model** to predict a quantitative variable from other quantitative
variables. Key steps: **learning** the model, **testing** it (performance + assumptions), and
**predicting** on new data.

We use the diabetes dataset again, building (1) a simple linear regression with one predictor, and
(2) a multiple regression with several predictors — predicting **cholesterol level**.

## 2 – Load the data and first analysis

In [ ]:
tmp = pd.read_csv("https://www.dropbox.com/s/zviurze7c85quyw/diabetes_full.csv?dl=1", sep="\t")

In [ ]:
# Limit the dataset to numerical variables
numeric_cols = ["chol", "stab.glu", "hdl", "glyhb", "age", "height", "weight", "bp.1s", "bp.1d", "waist", "hip"]
dat = tmp[numeric_cols]
dat.head()

First, remove all patients with at least 1 missing value:

In [ ]:
# R: i.na <- which(apply(dat, 1, function(x) sum(is.na(x)) > 0)); dat = dat[-i.na,]
dat = dat.dropna()
dat.shape

Let's look at the correlation between variables with scatter plots:

In [ ]:
# R: pairs(dat, col='red', pch=20, cex=0.5)
pd.plotting.scatter_matrix(dat, figsize=(14, 12), s=5, color="red")
plt.show()

Now a heatmap of the correlation values:

In [ ]:
# R: cor = cor(dat); pheatmap(cor, cluster_cols=FALSE, cluster_rows=FALSE, display_numbers=TRUE)
cor = dat.corr()
plt.figure(figsize=(9, 7))
sns.heatmap(cor, annot=True, fmt=".2f", cmap="vlag")
plt.show()

# What are the strongest correlations? Do they make sense?

There seems to be a positive correlation between stabilized glucose (`stab.glu`) and hip circumference (`hip`):

In [ ]:
from scipy import stats

## compute correlation
print(dat["stab.glu"].corr(dat["hip"]))

## test for significance
result = stats.pearsonr(dat["stab.glu"], dat["hip"])
print(result)

# Read the output carefully and make sure you understand it. Check other pairs!

## 3 – Univariate linear regression

> What is the most promising variable to predict cholesterol level?

We use glycosylated hemoglobin (`glyhb`) as a predictor. `statsmodels`' `ols()` formula syntax
mirrors R's `lm()` almost exactly:

In [ ]:
# R: l.g = lm(chol ~ glyhb, data=dat); summary(l.g)
l_g = smf.ols("chol ~ glyhb", data=dat).fit()
print(l_g.summary())

# For a SIMPLE regression (one predictor), the slope's t-test p-value equals the overall F-test
# p-value. This is no longer true once we add more predictors!

Have we checked that linear regression makes sense here? We need:

- residuals **normally distributed**
- **no correlation** between residuals and the explanatory variable

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(l_g.resid, bins=20)
axes[0].set_title("Residuals")
stats.probplot(l_g.resid, dist="norm", plot=axes[1])
plt.show()

## correlation residuals vs. x-values?
print(dat["glyhb"].corr(l_g.resid))

plt.scatter(dat["glyhb"], l_g.resid, s=10)
plt.xlabel("glyhb")
plt.ylabel("residuals")
plt.show()

# What is your overall opinion about the validity of this regression model?

Let's compare predictions with the real cholesterol values:

In [ ]:
plt.scatter(dat["chol"], l_g.fittedvalues, s=15, color="blue")
lims = [dat["chol"].min(), dat["chol"].max()]
plt.plot(lims, lims, color="red")
plt.xlabel("Real values")
plt.ylabel("Predicted values")
plt.show()

# Not super convincing, right? Let's add more information to the model!

## 4 – Multiple regression model

Let's include all variables to try to predict cholesterol:

In [ ]:
# R: l.all = lm(chol ~ ., data=dat); summary(l.all)
# "." means "all other columns" in R's formula syntax; statsmodels needs them spelled out
predictors_all = [c for c in dat.columns if c != "chol"]
formula_all = "chol ~ " + " + ".join(predictors_all)

l_all = smf.ols(formula_all, data=dat).fit()
print(l_all.summary())

# Do you note anything unexpected in this report?

Adding several explanatory variables improves the regression — check R² above. But `weight`,
`waist` and `hip` don't reach significance. Two explanations:

1. these variables really are non-informative for predicting cholesterol
2. mutual correlation between these 3 variables interferes with the model (multicollinearity)

Let's remove `waist` and `hip` and redo the regression:

In [ ]:
l_less = smf.ols("chol ~ Q('stab.glu') + hdl + glyhb + age + height + weight + Q('bp.1s') + Q('bp.1d')", data=dat).fit()
print(l_less.summary())

# weight now contributes more clearly — removing the correlated variables increased its
# significance. It's now much closer to the 5% level!
# Check the F-statistic to compare both models.

Are predictions better than the univariate model?

In [ ]:
plt.scatter(dat["chol"], l_less.fittedvalues, s=15, color="blue")
lims = [dat["chol"].min(), dat["chol"].max()]
plt.plot(lims, lims, color="red")
plt.xlabel("Real values")
plt.ylabel("Predicted values")
plt.show()

# Better? I would say so...

To quantify accuracy, we compute the **root mean squared error (RMSE)**:

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^n (x_i-\hat{x_i})^2}$$

In [ ]:
n = len(dat)
rmse = np.sqrt((l_less.resid**2).sum() / n)
rmse

Of course, this is a bit of cheating — we're predicting on the *same* data used to fit the model.
In real machine learning we need **cross-validation**: fit on a *training set*, evaluate on a
held-out *test set*.

In [ ]:
rng = np.random.default_rng(0)

## take 200 random patients to form the training set
i_train = rng.choice(dat.index, size=200, replace=False)
dat_train = dat.loc[i_train]
dat_test = dat.drop(index=i_train)

In [ ]:
l_train = smf.ols(
    "chol ~ Q('stab.glu') + hdl + glyhb + age + height + weight + Q('bp.1s') + Q('bp.1d')",
    data=dat_train
).fit()
print(l_train.summary())

In [ ]:
n_train = len(dat_train)
rmse_train = np.sqrt((l_train.resid**2).sum() / n_train)
rmse_train

Use the trained model to predict cholesterol on the held-out test set:

In [ ]:
# R: predict(l.train, newdata = dat.test)
pred = l_train.predict(dat_test)

In [ ]:
n_test = len(dat_test)
residuals_test = dat_test["chol"] - pred
rmse_test = np.sqrt((residuals_test**2).sum() / n_test)
rmse_test

# RMSE is higher on the test set — expected, since it wasn't used to fit the model.
# This is a more realistic estimate of how well the model generalizes to new data.

An important related topic is **feature selection**: finding the minimal, optimal set of
predictors. Let's repeat the train/test split 10 times with different random splits, and compare
train vs. test RMSE each time:

In [ ]:
rng = np.random.default_rng(345)

rmse_train_list = []
rmse_test_list = []

for _ in range(10):
    i_train = rng.choice(dat.index, size=200, replace=False)
    dat_train = dat.loc[i_train]
    dat_test = dat.drop(index=i_train)

    l_train = smf.ols(
        "chol ~ Q('stab.glu') + hdl + glyhb + age + height + weight + Q('bp.1s') + Q('bp.1d')",
        data=dat_train
    ).fit()

    n_train = len(dat_train)
    rmse_train_list.append(np.sqrt((l_train.resid**2).sum() / n_train))

    pred = l_train.predict(dat_test)
    n_test = len(dat_test)
    residuals_test = dat_test["chol"] - pred
    rmse_test_list.append(np.sqrt((residuals_test**2).sum() / n_test))

plt.scatter(range(10), rmse_train_list, color="orange", label="rmse.train")
plt.scatter(range(10), rmse_test_list, color="purple", label="rmse.test")
plt.axhline(np.mean(rmse_train_list), color="orange", linestyle="--")
plt.axhline(np.mean(rmse_test_list), color="purple", linestyle="--")
plt.xlabel("Iteration")
plt.ylabel("RMSE values")
plt.legend()
plt.show()

---
## Exercise

Take the multiple regression model and try removing unimportant variables step by step. Can you
identify an optimal set of variables? Which criterion would you use for that (p-values? R²? RMSE
on a test set?)

In [ ]:
# Your code here:

---
## Going further: using PCA to determine independent variables

To solve the correlated-variables problem, we compute principal components on all explanatory
variables:

In [ ]:
# R: pca = prcomp(dat[,-1])  — remove "chol" (the output variable), col 1 in R (1-indexed)
X = dat.drop(columns=["chol"])
pca = PCA()
pca_scores = pca.fit_transform(X)

pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(len(pca.explained_variance_ratio_))],
    "proportion_of_variance": pca.explained_variance_ratio_
})

We can display how PC1 is defined in terms of the original variables (its loadings):

In [ ]:
loadings_pc1 = pd.Series(pca.components_[0], index=X.columns).sort_values()
plt.barh(loadings_pc1.index, loadings_pc1.values, color="red")
plt.title("PC1")
plt.show()

Now run the multiple regression using the PCs as predictors instead of the original variables. First, check the PCs are indeed uncorrelated with each other:

In [ ]:
cor_pca = pd.DataFrame(pca_scores).corr()
plt.figure(figsize=(9, 7))
sns.heatmap(cor_pca, annot=True, fmt=".2f", cmap="vlag")
plt.show()

In [ ]:
# R: l.pca = lm(dat$chol ~ pca$x); summary(l.pca)
pca_df = pd.DataFrame(pca_scores, columns=[f"PC{i+1}" for i in range(pca_scores.shape[1])], index=X.index)
pca_df["chol"] = dat["chol"].values

formula_pca = "chol ~ " + " + ".join(pca_df.columns[:-1])
l_pca = smf.ols(formula_pca, data=pca_df).fit()
print(l_pca.summary())

# Produce barplots for the significant PCs, as we did for PC1 above.

Reproduce this analysis for the variable `glyhb` instead of `chol`:

In [ ]:
formula_glyhb = "glyhb ~ " + " + ".join([c for c in dat.columns if c != "glyhb"]).replace(
    "stab.glu", "Q('stab.glu')").replace("bp.1s", "Q('bp.1s')").replace("bp.1d", "Q('bp.1d')")
l_glyhb = smf.ols(formula_glyhb, data=dat).fit()
print(l_glyhb.summary())

X_glyhb = dat.drop(columns=["glyhb"])
pca_glyhb = PCA()
pca_glyhb_scores = pca_glyhb.fit_transform(X_glyhb)
print(pca_glyhb.explained_variance_ratio_)

cor_pca_glyhb = pd.DataFrame(pca_glyhb_scores).corr()
plt.figure(figsize=(9, 7))
sns.heatmap(cor_pca_glyhb, annot=True, fmt=".2f", cmap="vlag")
plt.show()

pca_glyhb_df = pd.DataFrame(pca_glyhb_scores, columns=[f"PC{i+1}" for i in range(pca_glyhb_scores.shape[1])], index=X_glyhb.index)
pca_glyhb_df["glyhb"] = dat["glyhb"].values
formula_pca_glyhb = "glyhb ~ " + " + ".join(pca_glyhb_df.columns[:-1])
l_pca_glyhb = smf.ols(formula_pca_glyhb, data=pca_glyhb_df).fit()
print(l_pca_glyhb.summary())

## Summary: What have we learned?

| R | Python | Purpose |
|---|--------|---------|
| `lm(y ~ x, data=df)` | `smf.ols("y ~ x", data=df).fit()` | Fit a linear model |
| `lm(y ~ ., data=df)` | `"y ~ " + " + ".join(other_cols)` | All-other-columns formula |
| `summary(model)` | `print(model.summary())` | Coefficients, p-values, R², F-test |
| `model$residuals` | `model.resid` | Residuals |
| `model$fitted.values` | `model.fittedvalues` | Fitted (predicted) values |
| `predict(model, newdata=df)` | `model.predict(df)` | Predict on new data |
| `cor.test(x, y)` | `scipy.stats.pearsonr(x, y)` | Correlation + significance test |
| `prcomp(df)` | `sklearn.decomposition.PCA().fit_transform(df)` | PCA |
| `pca$rotation[,1]` | `pca.components_[0]` | PC1 loadings |
| `pca$x` | `pca.fit_transform(df)` (the scores) | PCA scores |
| Column names with dots (`bp.1s`) in a formula | wrap in `Q('bp.1s')` | Escape special characters in patsy formulas |